In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Kruskal's Algorithm for Computing the Minimum Spanning Tree

In our implementation of [Kruskal's algorithm](https://en.wikipedia.org/wiki/Kruskal%27s_algorithm) for finding the 
*minimum spanning tree* we use the *union-find* data structure that we have defined previously.

In [ ]:
import { Node, union } from './UnionFindOO';

import { Graphviz } from "@hpcc-js/wasm";
import { display } from "tslab"
import * as fs from 'fs';

Furthermore, we need a priority queue. The module `heap-js` implements a priority queue.  The part
of the API from this module that we utilize is the following:
- `H.heappush(x)` pushes `x` onto the heap `H`,
- `H.heappop()`   removes the smallest element from the heap `H` and returns this element,
- `H = []`        creates an empty heap.

In [ ]:
import { Heap } from "heap-js";

The function $\texttt{mst}(V, E)$ takes a set of *nodes* $V$ and a set of *weighted edges* $E$ to compute a minimum spanning tree.  It is assumed that the pair $(V, E)$ represents a *weighted graph* $G$ that is *connected*.  The weighted edges in the set $E$ have the form
$$ \bigl\langle w, \langle x, y\rangle\bigr\rangle.  $$
Here, $x$ and $y$ are nodes from the set $V$, while $w$ is the cost of the edge $\{x,y\}$.  

The function call `mst(V, E)` returns a set of weighted edges that define a [minimum spanning tree](https://en.wikipedia.org/wiki/Minimum_spanning_tree)
of the weighted graph $G$. The function `mst` does **not** check whether $G$ is connected.

In [ ]:
type Edge = [number, [number, number]];

function mst(V: number[], E: Edge[]): Set<Edge> | null {

    const Nodes: { [x: number]: Node } = {};
    for (const x of V) {
        Nodes[x] = new Node(x);
    }

    const MST: Set<Edge> = new Set(); // minimum spanning tree, represented as set of weighted edges
    const H: Edge[] = E.slice().sort((a, b) => a[0] - b[0]); // empty priority queue for weighted edges

    while (true) {
        if (H.length === 0) {
            return null; // the graph is not connected
        }
        const edge = H.shift()!;
        const [w, [x, y]] = edge;
        const root_x = Nodes[x].find();
        const root_y = Nodes[y].find();
        if (root_x !== root_y) {
            MST.add(edge);
            union(Nodes[x], Nodes[y]);
            if (MST.size === V.length - 1) {
                return MST;
            }
        }
    }
}

The implementation of `mst` that is given below traces its computation via `Graphviz`.

In [ ]:
async function mst(V: number[], E: Edge[]): Promise<Set<Edge> | null> {
    const graphviz = await Graphviz.load();

    const initialDot = toDot(E);
    const initialSvg = graphviz.layout(initialDot, "svg", "dot");
    display.html(initialSvg);

    const Nodes: { [x: number]: Node } = {};
    for (const x of V) {
        Nodes[x] = new Node(x);
    }

    const MST: Set<Edge> = new Set(); // minimum spanning tree, represented as set of weighted edges
    const H: Edge[] = E.slice().sort((a, b) => a[0] - b[0]); // empty priority queue for weighted edges

    while (true) {
        if (H.length === 0) {
            return null; // the graph is not connected
        }
        const edge = H.shift()!;
        const [w, [x, y]] = edge;

        console.log(`testing ${x} - ${y}, weight ${w}`);

        const root_x = Nodes[x].find();
        const root_y = Nodes[y].find();

        if (root_x !== root_y) {
            console.log(`connect ${x} - ${y}`);
            MST.add(edge);
            union(Nodes[x], Nodes[y]);

            const mstDot = toDot(E, MST);
            const mstSvg = graphviz.layout(mstDot, "svg", "dot");
            display.html(mstSvg);

            console.log('_'.repeat(120));
            if (MST.size === V.length - 1) {
                return MST;
            }
        }
    }
}

Given a set $E$ of weighted edges, the function $\texttt{toDot}$ transforms this set into a dot structure that can be displayed as a graph.  The edges that are present in the set $H$ are assumed to be the edges that are part of the minimum spanning tree and therefore are highlighted.

In [ ]:
function toDot(E: Edge[], MST: Set<Edge> = new Set()): string {
    const V = new Set<number>();
    for (const [, [x, y]] of E) {
        V.add(x);
        V.add(y);
    }

    let dot = 'graph G {\n  rankdir=LR;\n';
    for (const x of V) {
        dot += `  ${x};\n`;
    }
    for (const [w, [x, y]] of E) {
        let inMST = false;
        for (const [mw, [mx, my]] of MST) {
            if (w === mw && ((x === mx && y === my) || (x === my && y === mx))) {
                inMST = true;
                break;
            }
        }
        if (inMST)
            dot += `  ${x} -- ${y} [label="${w}", color="blue", penwidth=2];\n`;
        else
            dot += `  ${x} -- ${y} [label="${w}", style="dashed"];\n`;
    }
    dot += '}';
    return dot;
}

The file `tiny.txt` contains the description of a weighted graph.  Every line in this file has the form:
```
x y w
```
Here `x` and `y` are numbers specifying nodes, while `w` is the weight of this node.  The code given below displays the file `tiny.txt`. 

In [ ]:
const s = fs.readFileSync('tiny.txt', 'utf-8');
console.log(s);

The function `demoFile(fn)` takes a filename `fn` as its argument. The corresponding file is expected to hold the description of
an *undirected weighted graph*.  The function computes the minimum spanning tree for this graph. 

In [ ]:
async function demoFile(fn: string): Promise<string | undefined> {
    const text = await fs.promises.readFile(fn, 'utf-8');
    const data = text.split('\n').filter(line => line.trim() !== '');

    const Edges: Set<Edge> = new Set();
    const Nodes: Set<number> = new Set();

    for (const line of data) {
        const [xStr, yStr, weightStr] = line.split(/\s+/);
        const x = parseInt(xStr), y = parseInt(yStr), weight = parseInt(weightStr);
        Edges.add([weight, [x, y]]);
        Nodes.add(x);
        Nodes.add(y);
    }

    const NodesArr = Array.from(Nodes);
    const EdgesArr = Array.from(Edges);
    const MST = await mst(NodesArr, EdgesArr);
    console.log(MST);

    if (MST) {
        return toDot(EdgesArr, MST);
    }
}

In [ ]:
const MSTDot = await demoFile('tiny.txt');
MSTDot